# Projeto (Seção 7) — solução do instrutor

## Especificação

`BoundedString` (2–50 caracteres) nos campos de texto; `BoundedList[T]` (1–5 elementos) com variável de tipo; novo campo
`top_features: BoundedList[BoundedString] | None = None` antes de `vin`. Enunciado em [`05_projeto`](05_projeto.ipynb).

In [1]:
from datetime import date
from enum import Enum
from uuid import UUID, uuid4
from pydantic import BaseModel, ConfigDict, Field, UUID4, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"

## Solução

In [2]:
from typing import Annotated, TypeVar
from pydantic import Field

BoundedString = Annotated[str, Field(min_length=2, max_length=50)]

T = TypeVar('T')

BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]

Testando os tipos antes de usá-los no modelo:

In [3]:
class Test(BaseModel):
    field1: BoundedString

In [4]:
print(f'Test(field1="abc"): {Test(field1="abc")!r}')

Test(field1="abc"): Test(field1='abc')


In [5]:
from pydantic import ValidationError

In [6]:
try:
    Test(field1="a")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
field1
  String should have at least 2 characters [type=string_too_short, input_value='a', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


In [7]:
try:
    Test(field1="a" * 51)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
field1
  String should have at most 50 characters [type=string_too_long, input_value='aaaaaaaaaaaaaaaaaaaaaaaa...aaaaaaaaaaaaaaaaaaaaaaa', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_long


In [8]:
class Test(BaseModel):
    my_list: BoundedList[int]

In [9]:
print(f"Test(my_list=[1, 2, 3]): {Test(my_list=[1, 2, 3])!r}")

Test(my_list=[1, 2, 3]): Test(my_list=[1, 2, 3])


In [10]:
try:
    Test(my_list=[])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
my_list
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


In [11]:
try:
    Test(my_list=[1, 2, 3, 4, 5, 6])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
my_list
  List should have at most 5 items after validation, not 6 [type=too_long, input_value=[1, 2, 3, 4, 5, 6], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


Agora combinamos os dois numa lista limitada de strings limitadas:

In [12]:
class Test(BaseModel):
    my_list: BoundedList[BoundedString]

In [13]:
print(f"Test(my_list=['aa', 'bb', 'cc']): {Test(my_list=['aa', 'bb', 'cc'])!r}")

Test(my_list=['aa', 'bb', 'cc']): Test(my_list=['aa', 'bb', 'cc'])


In [14]:
try:
    Test(my_list=[])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
my_list
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


In [15]:
try:
    Test(my_list=['a', 'bb', 'cc'])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
my_list.0
  String should have at least 2 characters [type=string_too_short, input_value='a', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


In [16]:
try:
    Test(my_list=['a' * 51, 'bb', 'cc'])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Test
my_list.0
  String should have at most 50 characters [type=string_too_long, input_value='aaaaaaaaaaaaaaaaaaaaaaaa...aaaaaaaaaaaaaaaaaaaaaaa', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_long


Funcionam; agora usamos no modelo:

In [17]:
class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD"
    )
    top_features: BoundedList[BoundedString] | None = None
    vin: BoundedString
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: BoundedString | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

> 📝 **Nota de revisão:** nesta solução o `id_` voltou a ser `UUID4 | None` — uma regressão do requisito da seção 6 ("`id` não deve mais
> ser anulável"). Com `| None`, um payload com `"id": null` passa. A solução comentada mantém `UUID4` sem `None`.

In [18]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [19]:
car = Automobile.model_validate(data)
print(f"car: {car!r}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>, is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, top_features=['6 cylinders', 'all-wheel drive', 'convertible'], vin='1234567890', number_of_doors=2, registration_country='France', license_plate='AAA-BBB')


In [20]:
assert car.model_dump(by_alias=True) == expected_serialized_by_alias

In [21]:
print(f'Automobile.model_validate({{**data, "id": None}}).id_: {Automobile.model_validate({**data, "id": None}).id_!r}')          # a regressão em ação

Automobile.model_validate({**data, "id": None}).id_: None
